# GAGA — Group AGgregation enhanced TrAnsformer for Graph Fraud Detection

### Neural Networks — Project Report (Colab notebook)

**Author:** Alessandro Nese — *1990771*  
**Date:** June 2026

---

A clean, dependency-light re-implementation and analysis of **GAGA**,
trained on the **Amazon** and **YelpChi** review graphs for graph-based
fraud detection. The notebook runs end-to-end on a Colab GPU.

## 1. Project Aim & Selected Paper

### Selected paper

> **Label Information Enhanced Fraud Detection against Low Homophily in Graphs**  
> Yuchen Wang, Jinghui Zhang, Zhengjie Huang, Weibin Li, Shikun Feng, Ziheng Ma,
> Yu Sun, Dianhai Yu, Fang Dong, Jiahui Jin, Beilun Wang, Junzhou Luo.  
> *The Web Conference (WWW), 2023.* (`GAGA.pdf` in this folder.)

### What the paper proposes

Graph-based fraud detection is a **node-classification** problem on
multi-relational graphs. Most GNN fraud detectors silently assume **homophily**
(connected nodes share a label) and break down under **low homophily**, which is
exactly the regime fraud lives in: fraudsters *camouflage* by connecting to many
benign nodes, so naive neighbourhood averaging buries the fraud signal in the
benign majority. The paper also notes that **label information** — known to help
node classification — is usually underused (labels are only a supervision signal,
never an input feature).

**GAGA** addresses both with two core ideas:

1. **Group aggregation** — instead of pooling a node's neighbours into one vector,
   split them by label (*benign / fraud / unknown*) and pool each group separately,
   so opposite-class signals stop cancelling out.
2. **Learnable group/hop/relation encodings + a Transformer encoder** — the group
   index *is* the class label, so adding a learnable group encoding injects label
   information directly into the feature space; a vanilla Transformer then
   re-weights the resulting short sequence.

The authors report up to **+24.39%** over competitive fraud detectors on YelpChi,
Amazon and an industrial Baidu dataset.

### Aim of this project

- Re-implement GAGA from scratch with a **light dependency stack** (PyTorch + SciPy,
  *no DGL*), keeping the pipeline transparent and reproducible.
- Reproduce the paper's headline behaviour on the two public datasets (Amazon, YelpChi).
- Analyse the training dynamics, metrics and limitations of the approach.

## 2. Theoretical Background & Key Concepts

### 2.1 Low homophily — why standard GNNs fail

Homophily is measured by the **edge homophily ratio** $h$, the fraction of edges
whose endpoints share a class:

$$ h = \frac{\big|\{(u,v)\in E : y_u = y_v\}\big|}{|E|} $$

High $h\to 1$ (e.g. citation graphs) is what message-passing GNNs are built for:
they **smooth** each node toward its neighbours. Under **low homophily** ($h\to 0$)
this smoothing mixes in other-class features and *dilutes* the signal — a fraud node
averaged toward its mostly-benign neighbourhood becomes indistinguishable from benign.

> **Subtlety (shown in §5).** Because benign nodes dominate (~93% on Amazon), the
> *global* edge homophily looks high. The relevant quantity is **class-conditional**:
> a *fraud* node's neighbours are overwhelmingly benign — that is the camouflage GAGA
> is built to defeat.

### 2.2 Problem setup

A graph $\mathcal{G}(V,E,X,Y)$ with $N$ nodes, $R$ relations (one adjacency per
relation), $d$-dimensional features, and **only a small set of labelled nodes**
(semi-supervised). Labels are binary: $y_v=1$ fraud, $y_v=0$ benign ($C=2$).

GAGA targets two weaknesses of prior work:
- **Challenge 1 — indiscriminate aggregation:** message passing mixes all neighbours
  regardless of label → hurts under low homophily. *Fixed by group aggregation.*
- **Challenge 2 — labels underused:** labels are only supervision, never input.
  *Fixed by the group encoding (label injection).*

### 2.3 The GAGA pipeline (five stages)

**1-2. Group aggregation (weight-free preprocessing, done once).**
Start from a stripped GNN aggregator — just the normalised neighbour sum on raw
features. Then **partition the sum by label**: with $C=2$ the groups are
$V^-$ (benign), $V^+$ (fraud) and $V^*$ (masked/unknown). Each group is pooled
separately:

$$ h_i = \frac{1}{\phi(\cdot)} \sum_{u \in V_i} x_u, \qquad \phi(\cdot)=|V_i|^{\alpha} $$

The target node is masked into $V^*$ to prevent **label leakage**. Running this per
hop $k=1\dots K$ and per relation, then prepending the node's own raw feature $h_v$,
produces a flat input sequence $H_s$ of length

$$ S = R \times (P\times K + 1), \qquad P = C+1 = 3. $$

For Amazon/YelpChi ($R=3, K=2, P=3$): **$S = 3\times(3\cdot 2+1) = 21$ tokens.**

**3. Linear projection.** Each of the $S$ group vectors is lifted from feature dim
$d$ to the Transformer hidden dim $d_H$: $X_s = \sigma(\psi(H_s))$.

**4. Learnable encodings.** A Transformer sees its input as an unordered set, so
three learnable lookup tables stamp each token with *where it came from* — and are
summed elementwise onto $X_s$:
- **Hop encoding** $X_h$ — which hop ($0=$ target, $1\dots K$).
- **Relation encoding** $X_r$ — which relation block.
- **Group encoding** $X_g$ — which label group; **this is the label-injection trick**,
  since the group index equals the class label.

$$ X_{in} = X_s + X_h + X_r + X_g $$

**5. Transformer encoder + readout + prediction.** A vanilla multi-head
self-attention encoder ($L$ layers) re-weights the sequence. GAGA then takes only the
**first (center) token of each relation block** — a BERT-`[CLS]`-style summary that
has absorbed its block — concatenates them across relations into $z_v$, and an MLP
head produces the fraud probability. Training uses (binary) cross-entropy over
labelled nodes only, plus L2 weight decay.

In [ ]:
# (Optional) show the architecture figure from the accompanying notes, if available.
import os
from IPython.display import Image, display
fig = 'GAGA Notion/image.png'
display(Image(fig)) if os.path.exists(fig) else print('Architecture figure not found (skipping).')

## 3. Implementation Details

### 3.1 Datasets

Both graphs ship as MATLAB `.mat` files (features, labels, one sparse adjacency per
relation) and are downloaded straight from the DGL mirror — no DGL dependency, graphs
are built with SciPy sparse matrices. The exact **seed-717** benchmark split is used
(40% train / 50% test / 10% val) so numbers stay comparable to the paper.

| Dataset | Nodes | Feat. dim | Relations | Fraud % | Notes |
|---------|-------|-----------|-----------|---------|-------|
| **Amazon** | 11,944 | 25 | 3 (`U-P-U`, `U-S-U`, `U-V-U`) | ~6.9% | first 3,305 nodes unlabelled, excluded from splits |
| **YelpChi** | 45,954 | 32 | 3 (`R-S-R`, `R-T-R`, `R-U-R`) | ~14.5% | review-spam graph |

Features are **row-normalised**. Both graphs are **low-homophily** (from the fraud
node's perspective), which is the setting GAGA is designed for — visualised in §5.

### 3.2 Graph → sequence (the heart of GAGA)

For every node, `gaga/sequence.py` walks the $K$-hop neighbourhood per relation and
summarises each hop into three group means using **only training labels** (a node
never sees its own label → no leakage):

```
[ center | hop1(benign, fraud, unknown) | hop2(...) ]   x  R relations   ->  (N, 21, d)
```

This is the one expensive, **one-off** step → cached to `seq_cache/` and reused.

### 3.3 Model (`gaga/model.py`)

- `SequenceEncoder`: linear feature projection (+ReLU) followed by three additive
  `nn.Embedding` tables (hop / relation / group), indexed by fixed per-token patterns
  registered as buffers.
- `nn.TransformerEncoder` of `n_layers` standard layers (`batch_first`).
- Readout: take the center token of every relation (stride `1 + K*(C+1)`),
  **concatenate** across relations, classify with a linear head.
- Xavier init; the Amazon model is ~tens of thousands of parameters (printed at train time).

> **Note / deviation from the paper.** This re-implementation uses a 2-logit head with
> `CrossEntropyLoss` (softmax) plus PR-curve **threshold-moving** at test time, rather
> than the paper's single-sigmoid BCE. Behaviour is equivalent for ranking metrics.

### 3.4 Experimental setup

Optimiser **Adam**, early stopping on **validation AUC** (patience 30), best-val
checkpoint saved. Key hyper-parameters per dataset:

| Config | emb_dim | layers | heads | ff_dim | dropout | lr | batch | hops |
|--------|---------|--------|-------|--------|---------|------|-------|------|
| `amazon.json` | 16 | 2 | 4 | 64 | 0.0 | 1e-3 | 256 | 2 |
| `yelp.json`   | 32 | 3 | 4 | 128 | 0.1 | 1e-3 | 512 | 2 |

Shared: `weight_decay=1e-4`, `max_epochs=500`, `early_stop=30`, `seed=717`,
`train_size=0.4`, `val_size=0.1`, features row-normalised.

## 4. Reproducibility & Colab Setup

**Steps:**
1. Set the runtime to GPU: *Runtime → Change runtime type → GPU*.
2. Copy this `Project/` folder into your Google Drive (e.g. `MyDrive/Project`).
3. Run the cells below top to bottom.

**Dependencies:** `torch` (ships with Colab), `scipy`, `scikit-learn`, `matplotlib`,
`tqdm`, `networkx` (for the graph plots) — full list in `requirements.txt`. The first
run downloads the datasets (cached in `data_cache/`) and builds the sequences (cached
in `seq_cache/`); later runs reuse both. The seed-717 split makes results deterministic.

In [ ]:
# 4.1 Check the GPU
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
# 4.2 Dependencies (torch ships with Colab; install the rest)
!pip install -q scipy scikit-learn matplotlib tqdm networkx

In [ ]:
# 4.3 Mount Drive and put the package on the path.
#     Adjust PROJECT_DIR if you stored the folder elsewhere.
import sys, os
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/Project'
sys.path.append(PROJECT_DIR)
os.chdir(PROJECT_DIR)
print('Working dir:', os.getcwd())

## 5. Exploratory Visualization of the Graphs

These graphs are far too large to draw in full (~12k / ~46k nodes, 3 relations each)
— a whole-graph layout is an unreadable *hairball*. Instead we use **targeted** views
that tie directly to GAGA's thesis:

1. **Homophily** — global vs. *class-conditional*, to expose fraud-node camouflage.
2. **Ego-network** of a single fraud node — the camouflage made visible.
3. **t-SNE** of raw features — how (in)separable the classes are *before* training.
4. **Adjacency spy plots** — sparsity/structure of each relation.

Everything below uses only `gaga.data.load_dataset` (features, labels, per-relation
CSR adjacencies). Set `DATASET` to `'amazon'` (fast) or `'yelp'`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from gaga.data import load_dataset

DATASET = 'amazon'   # 'amazon' (fast) or 'yelp'
REL_NAMES = {'amazon': ['U-P-U', 'U-S-U', 'U-V-U'],
             'yelp':   ['R-S-R', 'R-T-R', 'R-U-R']}[DATASET]

viz = load_dataset(DATASET, norm_feat=False)   # raw features for honest t-SNE
y = viz['labels']
adjs = [A.tocsr() for A in viz['adjacencies']]

# Labelled-node mask: the first 3305 Amazon nodes are unlabelled placeholders.
labelled = np.ones(len(y), dtype=bool)
if DATASET == 'amazon':
    labelled[:3305] = False
print(f'{DATASET}: nodes={len(y)}  labelled={labelled.sum()}  '
      f'fraud={int(y[labelled].sum())} ({y[labelled].mean():.1%})')

### 5.1 Homophily: global vs. class-conditional

The **global** edge homophily $h$ looks high simply because benign nodes dominate
(~93% on Amazon), so benign–benign edges swamp the count. The honest signal is
**class-conditional**: what fraction of a *fraud* node's neighbours are themselves
fraud? If that is low, fraud nodes are camouflaged among benign neighbours — exactly
the low-homophily regime GAGA is designed for.

In [ ]:
rows = []
for name, A in zip(REL_NAMES, adjs):
    coo = A.tocoo()
    m = (coo.row < coo.col) & labelled[coo.row] & labelled[coo.col]
    u, v = coo.row[m], coo.col[m]
    h_global = (y[u] == y[v]).mean()

    # class-conditional: among labelled neighbours, fraction that are fraud
    def frac_fraud_nbrs(node_class):
        nodes = np.where((y == node_class) & labelled)[0]
        tot = pos = 0
        for n in nodes:
            nb = A[n].indices; nb = nb[labelled[nb]]
            pos += int((y[nb] == 1).sum()); tot += len(nb)
        return pos / max(tot, 1)

    rows.append({'relation': name, 'edge_homophily': round(h_global, 3),
                 'fraud_nbrs_of_FRAUD': round(frac_fraud_nbrs(1), 3),
                 'fraud_nbrs_of_BENIGN': round(frac_fraud_nbrs(0), 3)})

import pandas as pd
homo = pd.DataFrame(rows).set_index('relation')
print(homo)

ax = homo[['fraud_nbrs_of_FRAUD', 'fraud_nbrs_of_BENIGN']].plot.bar(figsize=(7, 4))
ax.set_ylabel('fraction of neighbours that are FRAUD')
ax.set_title(f'{DATASET}: a fraud node\'s neighbours are mostly benign (camouflage)')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

### 5.2 Ego-network of a fraud node

One fraud node and its 1-hop neighbourhood under a single relation (capped for
readability), coloured **red = fraud / blue = benign**. The central fraud node
surrounded by blue is the camouflage that defeats plain neighbour-averaging.

In [ ]:
REL = 0                       # which relation to draw
MAX_NBRS = 40                 # cap neighbourhood size for readability
A = adjs[REL]
degs = np.asarray((A > 0).sum(1)).ravel()

# pick a labelled fraud node with a moderate degree so the plot is legible
cands = [n for n in np.where((y == 1) & labelled)[0] if 5 <= degs[n] <= 30]
center = cands[0]
nbrs = A[center].indices
nbrs = nbrs[nbrs != center][:MAX_NBRS]
nodes = np.r_[center, nbrs]

sub = A[nodes][:, nodes]
G = nx.from_scipy_sparse_array(sub)
colors = ['red' if y[n] == 1 else 'royalblue' for n in nodes]
sizes = [220] + [70] * (len(nodes) - 1)

plt.figure(figsize=(7, 6))
nx.draw_spring(G, node_color=colors, node_size=sizes, edge_color='lightgray',
               with_labels=False, seed=0)
plt.title(f'{DATASET} {REL_NAMES[REL]}: ego-net of fraud node {center} '
          f'({int((y[nbrs]==1).sum())}/{len(nbrs)} neighbours are fraud)')
plt.show()

### 5.3 t-SNE of raw node features

A 2-D t-SNE of (a sample of) the raw features, coloured by label. Heavy overlap here
is expected and is the whole point: **raw features alone do not separate fraud from
benign** — the model has to learn it. Re-run this on the learned embeddings $z_v$
after training (§7) for a *before vs. after* comparison, mirroring the paper's Fig. 4.

In [ ]:
from sklearn.manifold import TSNE

SAMPLE = 3000
idx = np.where(labelled)[0]
rng = np.random.RandomState(0)
idx = rng.choice(idx, min(SAMPLE, len(idx)), replace=False)

X = np.asarray(viz['features'])[idx]
emb = TSNE(n_components=2, init='pca', perplexity=30, random_state=0).fit_transform(X)

plt.figure(figsize=(6.5, 6))
for cls, col, lab in [(0, 'royalblue', 'benign'), (1, 'red', 'fraud')]:
    sel = y[idx] == cls
    plt.scatter(emb[sel, 0], emb[sel, 1], s=6, c=col, alpha=0.5, label=lab)
plt.legend(); plt.title(f'{DATASET}: t-SNE of raw features (n={len(idx)})')
plt.xticks([]); plt.yticks([]); plt.tight_layout(); plt.show()

### 5.4 Adjacency spy plots

A quick look at the sparsity pattern of each relation's adjacency matrix — relations
differ a lot in density (e.g. on Amazon, `U-S-U` is far denser than `U-P-U`), which is
why GAGA learns a separate **relation encoding** per relation.

In [ ]:
fig, axes = plt.subplots(1, len(adjs), figsize=(13, 4.3))
for ax, name, A in zip(axes, REL_NAMES, adjs):
    ax.spy(A, markersize=0.05, color='black')
    nnz = A.nnz
    ax.set_title(f'{name}  (nnz={nnz:,})')
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle(f'{DATASET}: per-relation adjacency sparsity'); plt.tight_layout(); plt.show()

## 6. Training

### 6.1 Amazon

`train(config)` downloads + builds sequences on the first run (then cached), trains
with early stopping, saves the best model to `checkpoints/amazon_best.pt`, and returns
the test metrics plus the per-epoch history.

In [ ]:
import json
from gaga.trainer import train, train_multiple

with open('configs/amazon.json') as f:
    config = json.load(f)
config

In [ ]:
test_metrics, history = train(config)

### 6.2 YelpChi

Larger graph (~46k nodes); a slightly bigger model (`emb_dim=32`, 3 layers).

In [ ]:
with open('configs/yelp.json') as f:
    yelp_config = json.load(f)

yelp_metrics, yelp_history = train(yelp_config)

## 7. Results & Analysis

### 7.1 Training curves

`plot_history` plots training loss (left axis) against validation AUC / F1-macro
(right axis). A healthy run shows loss decreasing while validation AUC climbs and
plateaus before early stopping triggers.

In [ ]:
from gaga.plot import plot_history
plot_history(history, save_path='checkpoints/amazon_curves.png', show=True)

In [ ]:
plot_history(yelp_history, save_path='checkpoints/yelp_curves.png', show=True)

### 7.2 Metrics table

We report the headline GAGA metrics on the held-out **test** set: **AUC**,
**F1-macro**, **G-Mean**, **AP** (average precision), and the fraud-class
precision/recall. The cell below collects them from both runs into one table.

In [ ]:
import pandas as pd

rows = {'Amazon': test_metrics, 'YelpChi': yelp_metrics}
cols = ['auc', 'f1_macro', 'gmean', 'ap', 'precision_1', 'recall_1', 'f1_fraud']
df = pd.DataFrame(rows).T[cols].round(4)
df.columns = ['AUC', 'F1-macro', 'GMean', 'AP', 'Precision(fraud)', 'Recall(fraud)', 'F1(fraud)']
df

### 7.3 Reference / reproduction targets

Same seed-717 split as the original benchmark; approximate expected results:

| Dataset | AUC (target) | F1-macro (target) |
|---------|-------------|-------------------|
| Amazon  | ~0.96 | ~0.91 |
| YelpChi | ~0.94 | ~0.90 |

### 7.4 Multiple runs (mean ± std)

Because the model is small and the split fixed, variance comes mainly from weight
initialisation and mini-batch order. `train_multiple` repeats training and reports
mean ± std of each metric — a more honest summary than a single run.

In [ ]:
_ = train_multiple(config, n_runs=5)

### 7.5 Interpretation

- **AUC vs F1.** Both datasets are **imbalanced** (Amazon ~6.9% fraud, YelpChi
  ~14.5%), so AUC and F1-macro matter more than raw accuracy. Threshold-moving on the
  validation PR curve is what lifts the fraud-class F1.
- **Why group aggregation helps.** §5 showed a fraud node's neighbours are mostly
  benign; splitting neighbours by label keeps the fraud and benign signals in separate
  tokens instead of averaging them away. The group encoding then lets attention treat
  "this is a fraud-group token" as a first-class feature.
- **Cost profile.** Aggregation is weight-free and cached; the trainable model is a
  small Transformer over only 21 tokens, so training is fast and memory-light
  relative to deep message-passing GNNs.

## 8. Limitations & Reflections

- **Binary, transductive setting.** The formulation assumes $C=2$ and a fixed graph
  with a known split; the group structure ($P=C+1$) would need rethinking for
  multi-class or fully inductive use. (*Generalising GAGA to multi-class is the
  paper's own stated future work.*)
- **Dependence on labelled neighbours.** Group means are computed from *training*
  labels only. When a node has few/no labelled neighbours, its benign/fraud tokens
  fall back to zeros and only the *unknown* group carries signal — quality degrades
  in sparsely-labelled regions.
- **Preprocessing cost & staleness.** Sequences are cached for one split; changing
  hops/split/seed forces a rebuild, and the cache must be invalidated carefully.
- **Implementation deviations.** Softmax 2-logit head + CrossEntropy + threshold
  moving instead of the paper's sigmoid-BCE; metrics are comparable but not bit-exact
  to the reference. The industrial BF10M dataset from the paper is not reproduced.
- **Reflections.** The project shows that *how* labels enter the model can matter more
  than model depth: a shallow Transformer over a cleverly grouped 21-token sequence
  matches deep GNNs in the low-homophily regime. The main engineering lessons were
  avoiding label leakage in the grouping step and making the expensive aggregation
  reproducible via on-disk caching.

## 9. References

**Papers**
1. Y. Wang, J. Zhang, Z. Huang, W. Li, S. Feng, Z. Ma, Y. Sun, D. Yu, F. Dong,
   J. Jin, B. Wang, J. Luo. *Label Information Enhanced Fraud Detection against Low
   Homophily in Graphs.* WWW 2023. (`GAGA.pdf`)
2. A. Vaswani et al. *Attention Is All You Need.* NeurIPS 2017.
3. W. Hamilton, R. Ying, J. Leskovec. *Inductive Representation Learning on Large
   Graphs (GraphSAGE).* NeurIPS 2017.
4. Y. Dou et al. *Enhancing Graph Neural Network-based Fraud Detectors against
   Camouflaged Fraudsters (CARE-GNN).* CIKM 2020. — source of the YelpChi/Amazon graphs.

**Code & datasets**
5. Original GAGA reference implementation — <https://github.com/Orion-wyc/GAGA>
6. YelpChi graph (`FraudYelp.zip`) — <https://data.dgl.ai/dataset/FraudYelp.zip>
7. Amazon graph (`FraudAmazon.zip`) — <https://data.dgl.ai/dataset/FraudAmazon.zip>
8. This project's source: `gaga/` (`data.py`, `sequence.py`, `model.py`,
   `trainer.py`, `metrics.py`, `plot.py`), configs in `configs/`.